In [10]:
-- S1 - SQL Foundations
-- using duckdb
-- duckdb is a modern process sql-engine that runs directly in python
-- you do not need to setup a server or install postgresql
-- duckdb will read pandas dataframes directly and support full sql syntax
import duckdub
import pandas as pd

-- result = duckdb.query("select * from df where amount > 5000").df()

-- basic sql structure
SELECT   -- what columns do you want?
FROM     -- which table?
WHERE    -- which rows?
GROUP BY -- how to group?
HAVING   -- filter after grouping?
ORDER BY -- how to sort?
LIMIT    -- how many rows?

-- execution order
-- Execution order:
1. FROM      -- get the table
2. WHERE     —- filter rows
3. GROUP BY  —- group the filtered rows
4. HAVING    —- filter the groups
5. SELECT    -- choose columns
6. ORDER BY  —- sort the result
7. LIMIT     —- cap the rows

SyntaxError: invalid character '—' (U+2014) (3749262096.py, line 23)

In [11]:
# build the supply chain dataset
import duckdb
import pandas as pd
import numpy as np

np.random.seed(22)

suppliers = ["Acme", "GlobalCo", "FastParts", "PrimeMg", "EastCoast"]
categories = ["Electronics", "Hardware", "Consumables", "Apparel"]
warehouses = ["East", "West", "Central"]

rows = []
for i in range(1, 201):
    rows.append({
        "po_id":          f"PO-{str(i).zfill(4)}",
        "supplier":       np.random.choice(suppliers),
        "category":       np.random.choice(categories),
        "warehouse":      np.random.choice(warehouses),
        "order_date":     pd.Timestamp("2024-01-01") + pd.Timedelta(days=int(np.random.randint(0, 365))),
        "amount":         round(np.random.uniform(500, 50000), 2),
        "units":          np.random.randint(10, 500),
        "lead_time_days": np.random.randint(3, 45),
        "on_time":        np.random.choice([True, False], p=[0.8, 0.2])
    })

po = pd.DataFrame(rows)
print(po.shape)
print(po.head())

(200, 9)
     po_id   supplier     category warehouse order_date    amount  units  \
0  PO-0001  EastCoast  Electronics      East 2024-12-22  32055.22     94   
1  PO-0002  FastParts  Consumables      East 2024-02-15  23470.66    103   
2  PO-0003       Acme  Electronics      West 2024-01-28  22382.64     39   
3  PO-0004   GlobalCo      Apparel   Central 2024-01-08  47315.60     33   
4  PO-0005  EastCoast      Apparel      East 2024-12-25  37347.63    123   

   lead_time_days  on_time  
0              11     True  
1              42     True  
2              21     True  
3              26     True  
4               4    False  


In [3]:
# S1.1 SELECT and FROM

# query 1 — select all columns, first 10 rows
# hint: use * to select all columns, LIMIT to cap rows

result = (
    duckdb.query("""
select 
    * 
from 
    po 
limit 
    10""")
.df()
)

print(result)

     po_id   supplier     category warehouse order_date    amount  units  \
0  PO-0001  EastCoast  Electronics      East 2024-12-22  32055.22     94   
1  PO-0002  FastParts  Consumables      East 2024-02-15  23470.66    103   
2  PO-0003       Acme  Electronics      West 2024-01-28  22382.64     39   
3  PO-0004   GlobalCo      Apparel   Central 2024-01-08  47315.60     33   
4  PO-0005  EastCoast      Apparel      East 2024-12-25  37347.63    123   
5  PO-0006    PrimeMg  Consumables      West 2024-03-01  27561.29    453   
6  PO-0007  EastCoast     Hardware      West 2024-12-06   9973.63    397   
7  PO-0008  EastCoast  Electronics   Central 2024-12-01  34588.33    426   
8  PO-0009       Acme     Hardware      East 2024-01-18  39190.33    418   
9  PO-0010  FastParts     Hardware      West 2024-01-06   8790.67    200   

   lead_time_days  on_time  
0              11     True  
1              42     True  
2              21     True  
3              26     True  
4               4 

In [12]:
# S1.1
# select only three specific columns
result = (duckdb.query("""
select
    po_id,
    order_date,
    amount
from
    po
limit
    10
"""
)
.df()
)
print(result)


     po_id order_date    amount
0  PO-0001 2024-12-22  32055.22
1  PO-0002 2024-02-15  23470.66
2  PO-0003 2024-01-28  22382.64
3  PO-0004 2024-01-08  47315.60
4  PO-0005 2024-12-25  37347.63
5  PO-0006 2024-03-01  27561.29
6  PO-0007 2024-12-06   9973.63
7  PO-0008 2024-12-01  34588.33
8  PO-0009 2024-01-18  39190.33
9  PO-0010 2024-01-06   8790.67


In [13]:
# S1.1
# query 3 - calculated column
result = (duckdb.query("""
                       select
                       po_id,
                       supplier,
                       amount,
                       units,
                       amount * units as inventory_value
                       from
                       po
                       limit 10"""
                       )
                       .df()
                       )
print(result)


     po_id   supplier    amount  units  inventory_value
0  PO-0001  EastCoast  32055.22     94       3013190.68
1  PO-0002  FastParts  23470.66    103       2417477.98
2  PO-0003       Acme  22382.64     39        872922.96
3  PO-0004   GlobalCo  47315.60     33       1561414.80
4  PO-0005  EastCoast  37347.63    123       4593758.49
5  PO-0006    PrimeMg  27561.29    453      12485264.37
6  PO-0007  EastCoast   9973.63    397       3959531.11
7  PO-0008  EastCoast  34588.33    426      14734628.58
8  PO-0009       Acme  39190.33    418      16381557.94
9  PO-0010  FastParts   8790.67    200       1758134.00


In [14]:
# S1.2
# using WHERE

# find po_id's with amount > 40,000
result = (
    duckdb.query("""
                 select
                 po_id,
                 supplier,
                 amount
                 from
                 po
                 where
                 amount > 40000
                 order by amount desc
                 """)
                 .df()
)
print(f"PO's over $40,000: {len(result)}")

PO's over $40,000: 36


In [15]:
# multiple conditions
# find electronics > 10K
result = (
    duckdb.query("""
select
                 po_id,
                 supplier,
                 category,
                 amount
from
                 po
where
                 category = 'Electronics'
and
                 amount > 10000                 
""")
.df()
)
print(f"\nElectronics PO's over $10K: {len(result)}")


Electronics PO's over $10K: 46


In [16]:
# in operator
# return the east and west warehouses only
result = (
    duckdb.query("""
                 select
                 po_id,
                 supplier,
                 warehouse
                 from
                 po
                 where
                 warehouse in ('East', 'West')
                 """)
                 .df()
)
print(f"\nEast and West PO's: {len(result)}")


East and West PO's: 130


In [17]:
# use BETWEEN
# amounts between 10k and 20k
result = duckdb.query(
    """
select
po_id,
supplier,
amount
from
po
where 
amount 
between 10000 and 20000
order by amount desc
    """
)
print(f"\nPO's between $10K and $20K: {len(result)}")


PO's between $10K and $20K: 41


In [25]:
# S1.3 GROUP BY
# group total spend by supplier sorted highest to lowest
import duckdb
import pandas as pd
import numpy as np
%load_ext sql


result = duckdb.query("""
select
 supplier,
count(*) as po_count,
round(sum(amount), 2) as total_spend,
round(avg(amount), 2) as avg_po_value
from po
group by supplier
order by total_spend desc
""").df()

print(result)


The sql extension is already loaded. To reload it, use:
  %reload_ext sql
    supplier  po_count  total_spend  avg_po_value
0  EastCoast        54   1200154.42      22225.08
1    PrimeMg        43   1071116.69      24909.69
2  FastParts        42    995411.84      23700.28
3       Acme        32    819898.26      25621.82
4   GlobalCo        29    752094.86      25934.31


In [37]:
%%sql

select
    supplier,
    round(sum(amount), 2) as total_spend
from
    po
group by 
    supplier
having 
    sum(amount) > 900000
order by 
    total_spend desc



Running query in 'duckdb:///:memory:'

supplier,total_spend
EastCoast,1200154.42
PrimeMg,1071116.69
FastParts,995411.84


In [23]:
print(po.head(5))

     po_id   supplier     category warehouse order_date    amount  units  \
0  PO-0001  EastCoast  Electronics      East 2024-12-22  32055.22     94   
1  PO-0002  FastParts  Consumables      East 2024-02-15  23470.66    103   
2  PO-0003       Acme  Electronics      West 2024-01-28  22382.64     39   
3  PO-0004   GlobalCo      Apparel   Central 2024-01-08  47315.60     33   
4  PO-0005  EastCoast      Apparel      East 2024-12-25  37347.63    123   

   lead_time_days  on_time  
0              11     True  
1              42     True  
2              21     True  
3              26     True  
4               4    False  


In [31]:
%load_ext sql
%sql duckdb:///:memory:
%sql --persist po

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


Running query in 'duckdb:///:memory:'

ValueError: Table 'po' already exists. Consider using --persist-replace to drop the table before persisting the data frame


In [33]:
%sql select count(*) as total_rows from po

Running query in 'duckdb:///:memory:'

total_rows
200
